In [ ]:
import tqdm
import numpy as np
from matplotlib import pyplot as plt

from skimage.feature import plot_matched_features
import cv2

import pykitti

In [ ]:
!wget -nc -q -O kitti_dataset.zip "https://www.dropbox.com/scl/fi/x4y2mwe53s17ar4w2xs1x/kitti_dataset.zip?rlkey=g1er6uwdi3g4s3awlbcfixk61&dl=1"
!unzip -n -q kitti_dataset.zip

In [ ]:
data = pykitti.odometry("kitti_dataset", "05")

# or, can restrict the number of frames during debugging like this:
# data = pykitti.odometry("kitti_dataset", "05", frames=range(20))

In [250]:
K_camL = data.calib.K_cam0           # intrinsics matrix for left camera
K_camR = data.calib.K_cam1           # intrinsics matrix for right camera
P_camL = data.calib.P_rect_00        # projection matrix for left camera
P_camR = data.calib.P_rect_10        # projection matrix for right camera

In [251]:
gt_poses = np.array(data.poses)      # camera-to-world transformation for each timestamp

In [252]:
stereo_pairs = [                     # stereo image pairs
    (np.array(L),np.array(R)) 
    for L,R in data.gray
]

In [253]:
def detect_and_extract(extractor,img):
    """ Detect keypoints and extract descriptors from a grayscale image. 
        Note that keypoints are stored as (x,y).
    """
    kps, descriptors = extractor.detectAndCompute(img, None)
    keypoints = np.array([p.pt for p in kps])
    return keypoints, descriptors

def match_descriptors(matcher,descriptors1,descriptors2):
    """ Match ORB descriptors and return skimage-style match array:
        first column is index into descriptors1, second column is index into descriptors2.
    """
    dmatches = matcher.match(descriptors1, descriptors2)
    matches = np.array([(m.queryIdx,m.trainIdx) for m in dmatches])
    return matches

In [254]:
# Create the ORB extractor and brute force matcher
extractor = cv2.ORB_create(nfeatures=2000)
matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)

In [ ]:
# YOUR CODE HERE